# Knowledge representation on medical dataset E3C using fine tuned BERT model


## 1. Introduzione e Obiettivo del Task
L'obiettivo di questo progetto è esplorare tecniche avanzate di **Knowledge Representation** applicate al dominio clinico. Nello specifico, il task affrontato è il **Named Entity Recognition (NER)**, formulato computazionalmente come un problema di *Token Classification*. 

Lo scopo del sistema è analizzare referti medici testuali non strutturati ed estrarre automaticamente i concetti di interesse clinico, identificando l'esatta posizione di patologie, sindromi e sintomi (etichettati nel sistema come `CLINENTITY`).

## 2. Il Dataset: European Clinical Case Corpus (E3C)
I dati utilizzati provengono dall'**E3C (European Clinical Case Corpus)**, specificamente dal *Layer 1* (Gold Standard), che contiene annotazioni cliniche validate manualmente da esperti del settore. 

Lavorare con testi medici in italiano presenta una sfida notevole (problema del *Low-Resource Domain*). Il dataset italiano del Layer 1 è costituito da circa 80 documenti. Come dimostreremo, questa scarsità di dati rende il modello addestrato non al massimo dell'ottimalità, causando fenomeni di sbilanciamento delle classi (collasso sull'etichetta `O` - Outside) specialmente nel caso di *Few-Shot Learning*.

## 3. Metodologia ed Esperimenti
Per analizzare e superare questi limiti strutturali, il progetto si articola in una serie di esperimenti a complessità crescente:

* **Baseline Monolingua (BERT Italiano):** Fine-tuning del modello `dbmdz/bert-base-italian-cased` sui soli dati italiani, valutando l'impatto della dimensione del dataset tramite esperimenti *One-Shot* (1 documento), *Few-Shot* (10 documenti) e *Full-Shot* (~80 documenti).
* **Cross-Lingual Transfer Learning:** Per mitigare la scarsità di dati italiani, è stato costruito un pool di addestramento multilingue, fondendo le annotazioni E3C in Italiano, Inglese, Spagnolo, Francese e Basco.
* **Architetture Multilingua (mBERT e XLM-RoBERTa):** Fine-tuning di modelli nativamente multilingua (`bert-base-multilingual-cased` e `xlm-roberta-base`) per dimostrare come i *Word Embeddings* permettano di trasferire la conoscenza semantica delle patologie tra lingue diverse, migliorando nettamente la metrica F1-Score.


# Conversione dati di E3C

I documenti presenti all'interno del dataset di riferimento sono in formato *XML* tuttavia la notazione non è quella classica di riferimento bensì quella *XMI* (XML Metadata Interchange), file di questo tipo provengono da WebAnno ovvero una piattaforma accademica utilizzata per l'annotazione linguistica di testi. I token vengono identificati da ID univoci e su di essi vengono definite le relazioni e le posizioni tra loro nel testo grezzo ovvero, quello contenuto nel tag `Sofa` (Subject of Analisys) presente alla fine del file.

é stata necessaria quindi la creazione di due script che portassero i nostri documenti in formato JSON ovvero `generate_dataset_json.py` e `generate_it_dataset_json.py`. 

Eccone il funzionamento:



In [1]:
from scripts.generate_it_dataset_json import parse_xmi_e3c
import os

XML_file = "data/raw/E3C-Corpus-2.0.0/data_annotation/Italian/layer1/IT100002.xml"

json_element = parse_xmi_e3c(XML_file)
print(json_element)


{'id_doc': 'IT100002.xml', 'text': 'Anna è una donna di 47 anni, vive con il figlio in un piccolo appartamento in un paese carsico, lavora nel campo della ristorazione e, nonostante una patologia genetica familiare che provoca delle alterazioni scheletriche, cardiologiche, oculari e cutanee, vive una vita serena e tranquilla. Ha una buona rete familiare e informale che supporta la famiglia. Viene seguita da un centro di riferimento specifico per patologia rara a Bologna e si sottopone a controlli clinicostrumentali periodici e regolari. Nel gennaio del 2017 esegue un intervento chirurgico di piede torto congenito presso l’Istituto Rizzoli. Durante la degenza, a seguito di un dolore retrosternale e di una sincope, viene sottoposta a un intervento di endoprotesi di aorta per dissezione acuta. Al rientro a casa, nel marzo del 2017, deve proseguire i controlli cardiologici ed eseguire la fisioterapia per gli esiti di intervento al piede e regolari controlli ematici. Non essendo autonoma ne

In [2]:
from scripts.generate_dataset_json import parse_xmi_e3c
import os 

XML_file = "data/raw/E3C-Corpus-2.0.0/data_annotation/English/layer1/EN100017.xml"

json_element = parse_xmi_e3c(XML_file)
print(json_element)

{'id_doc': 'EN100017.xml', 'lingua': 'English', 'text': "A 14-year old boy with no significant past medical history presented to a small district hospital in southern Sierra Leone with a 4 day history of facial puffiness, peripheral pitting oedema, abdominal pains, and reduced urine output. On examination he was afebrile, BP150/110, heart rate 70. He had significant periorbital and facial oedema, pitting oedema from the feet to the knees, a distended abdomen, ascites, and tender hepatomegaly. Lab results showed haemoglobin 10.9 g/dl, packed cell volume 35%, and positive malaria parasites. Urea (14 mg/dl), creatinine (1.0 mg/dl), sodium (137 mmol/L), and potassium (3.7 mmol/L) were normal. Urinalysis, using a urine dipstick, revealed three pluses of proteinurea, which equates to ≥3g urinary protein per day. Lab facilities for the measurement of serum albumin were not available. The diagnosis of nephrotic syndrome was made and the patient was started on a course of prednisolone 60mg/day,

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification


def file_to_string(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        return "Error: file not found."
    except Exception as e:
        return f"Unexpected error: {e}"


# --- 1. CONFIGURAZIONE ---
CARTELLA_MODELLO = "model/mul_bert_medico_full_shot" 
#MODEL_NAME = "dbmdz/bert-base-italian-cased"
MODEL_NAME = "bert-base-multilingual-cased"
#MODEL_NAME = "xlm-roberta-large"

print(f"Caricamento del modello da: {CARTELLA_MODELLO}...")

# Carichiamo il Tokenizer base e il TUO modello addestrato
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
modello = AutoModelForTokenClassification.from_pretrained(CARTELLA_MODELLO)

# Creiamo la "Pipeline" (è uno strumento di Hugging Face che fa tutto il lavoro sporco 
# di tokenizzazione, predizione e ri-allineamento delle parole)
# aggregation_strategy="simple" unisce in automatico i sub-token ("elettro" + "##cardio")
ner_pipeline = pipeline("token-classification", model=modello, tokenizer=tokenizer, aggregation_strategy="simple")

# --- 2. IL TEST DAL VIVO ---
print("\nScrivi un referto medico inventato (o premi Invio per usare l'esempio).")
print("Digita 'esci' per terminare.")

while True:
    testo_input = input("\nReferto: ")
    
    if testo_input.lower() == 'esci':
        break
        
    if not testo_input.strip():
        # predef input file
        testo_input = file_to_string("example/fr_example.txt")
        print(f"Uso l'esempio: {testo_input}")

    # Inference
    risultati = ner_pipeline(testo_input)
    
    print("\n--- Risultati ---")
    if not risultati:
        print("Nessuna entità clinica trovata.")
    else:
        for entita in risultati:
            parola = entita['word']
            etichetta = entita['entity_group']
            score = entita['score'] * 100
            
            print(f" Trovato: '{parola}'")
            print(f"   Tipo: {etichetta} (Score del modello: {score:.1f}%)\n")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# --- 1. CONFIGURAZIONE STILE ---
# Usiamo Seaborn per rendere il grafico molto più elegante e moderno
sns.set_theme(style="whitegrid", font_scale=1.1)

# --- 2. I TUOI DATI ---
# Questi sono i risultati esatti che abbiamo ottenuto durante l'esperimento
esperimenti = [
    "One-Shot\n(Italiano)", 
    "Few-Shot (10)\n(Italiano)", 
    "Full-Shot\n(Italiano)", 
    "mBERT\n(Multilingue)", 
    "XLM-RoBERTa\n(Multilingue)"
]

f1_scores = [0.00, 0.46, 0.60, 0.64, 0.67]

# Colori per indicare la progressione:
# Rosso (Fallimento), Azzurro (Miglioramento IT), Blu (Tetto IT), Arancione (mBERT), Verde (Vittoria Finale)
colori = ['#e74c3c', '#3498db', '#2980b9', '#f39c12', '#27ae60']

# --- 3. DISEGNO DEL GRAFICO ---
plt.figure(figsize=(11, 6)) # Dimensioni larghezza x altezza (perfette per le slide 16:9)

# Disegniamo le barre
bars = plt.bar(esperimenti, f1_scores, color=colori)

# --- 4. PERSONALIZZAZIONI ---
plt.ylim(0, 0.8) # Alziamo il limite Y per non tagliare i numeri in cima
plt.ylabel("Metrica F1-Score", fontsize=12, fontweight='bold')
plt.title("Evoluzione delle Performance NER Clinico (Knowledge Representation)", fontsize=16, fontweight='bold', pad=20)

# Aggiungiamo il valore esatto sopra ogni barra
for bar in bars:
    altezza = bar.get_height()
    # Scriviamo il numero formattato con 2 decimali
    plt.text(bar.get_x() + bar.get_width() / 2, altezza + 0.01, 
             f"{altezza:.2f}", 
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# Aggiungiamo una leggera linea tratteggiata per mostrare il trend di crescita
plt.plot(esperimenti, f1_scores, color='gray', linestyle='--', marker='o', alpha=0.5)

# --- 5. SALVATAGGIO ---
plt.tight_layout()
nome_file = 'grafico_risultati_esame.png'
plt.savefig(nome_file, dpi=300) # dpi=300 garantisce qualità da stampa tipografica
print(f"Grafico salvato con successo come: {nome_file}")